# Notebook A - Build Dataset

This notebook builds and validates the multimodal segmentation dataset used by training: RGB + Depth + Height + Mask.

Workflow:
1. Check source and target folders
2. Build dataset artifacts from nuScenes mini (optional if already built)
3. Create train/val/test split files
4. Validate dataset and dataloader output

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

def is_project_root(p: Path) -> bool:
    return (p / 'config' / 'config.py').exists() and (p / 'data' / 'dataset.py').exists()

def iter_dirs_bounded(base: Path, max_depth: int = 4):
    """Yield directories under base up to max_depth (bounded BFS)."""
    if not base.exists() or not base.is_dir():
        return
    queue = [(base, 0)]
    seen = set()

    while queue:
        current, depth = queue.pop(0)
        if current in seen:
            continue
        seen.add(current)
        yield current

        if depth >= max_depth:
            continue

        try:
            for child in current.iterdir():
                if child.is_dir():
                    queue.append((child, depth + 1))
        except PermissionError:
            continue

def find_project_root(start: Path) -> Path:
    """Find project root robustly for local, Kaggle, and hosted notebook runtimes."""
    # 1) Manual override first
    env_root = os.environ.get('PROJECT_ROOT')
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if is_project_root(p):
            return p

    # 2) CWD and parents
    start = start.resolve()
    for p in [start, *start.parents]:
        if is_project_root(p):
            return p

    # 3) Exact common candidates
    exact_candidates = [
        Path('/kaggle/working/Space-segmentation-model-MAHE'),
        Path('/kaggle/input/Space-segmentation-model-MAHE'),
        Path('/kaggle/input/space-segmentation-model-mahe'),
        Path('/kaggle/working'),
        Path('/kaggle/input'),
        Path('/content'),
        Path.home(),
        start / 'Space-segmentation-model-MAHE',
    ]
    for p in exact_candidates:
        if is_project_root(p):
            return p.resolve()

    # 4) Bounded recursive search in likely bases
    search_bases = [start, Path('/kaggle/working'), Path('/kaggle/input'), Path('/content')]
    for base in search_bases:
        for d in iter_dirs_bounded(base, max_depth=5):
            if is_project_root(d):
                return d.resolve()

    raise RuntimeError(
        "Could not locate project root automatically. "
        "Set PROJECT_ROOT manually, e.g.:\n"
        "import os\n"
        "os.environ['PROJECT_ROOT'] = '/kaggle/input/<your-dataset-folder>/Space-segmentation-model-MAHE'"
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import config
from data.preprocess_multi_thread import run_multimodal_preprocessing
from data.augmentations import get_train_transforms
from data.dataset import MultiModalDrivableDataset

print(f'Project root: {PROJECT_ROOT}')
print(f'Source nuScenes root: {config.DATA_ROOT}')
print(f'Target dataset root: {config.DATASET_DIR}')
print(f'Image size: {config.IMG_SIZE}')
print(f'Cameras: {config.CAMERAS}')

In [ ]:
def count_generated_images(dataset_dir, cameras):
    counts = {}
    for cam in cameras:
        img_dir = dataset_dir / cam / 'images'
        counts[cam] = len(list(img_dir.glob('*.png'))) if img_dir.exists() else 0
    return counts

def print_counts(counts):
    total = 0
    for cam, n in counts.items():
        print(f'{cam:16s} -> {n:4d} images')
        total += n
    print(f'Total camera-images: {total}')

assert config.DATA_ROOT.exists(), f'nuScenes source folder not found: {config.DATA_ROOT}'
counts_before = count_generated_images(config.DATASET_DIR, config.CAMERAS)
print('Current generated dataset counts:')
print_counts(counts_before)

## Step 2 - Build Dataset Artifacts

Set `FORCE_REBUILD = True` if you want to regenerate all camera outputs from nuScenes mini.
If your counts are already non-zero and look correct, keep it `False`.

In [ ]:
FORCE_REBUILD = False

if FORCE_REBUILD:
    run_multimodal_preprocessing()
    print('Preprocessing finished.')
else:
    print('Skipped preprocessing. Set FORCE_REBUILD=True to run it.')

counts_after = count_generated_images(config.DATASET_DIR, config.CAMERAS)
print('Dataset counts after step:')
print_counts(counts_after)

## Step 3 - Create/Refresh Splits

This creates `dataset/splits/train.txt`, `val.txt`, and `test.txt` using a deterministic seed.

In [ ]:
SPLIT_DIR = config.DATASET_DIR / 'splits'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8

def tokens_for_camera(cam):
    image_dir = config.DATASET_DIR / cam / 'images'
    return {p.stem for p in image_dir.glob('*.png')}

token_sets = [tokens_for_camera(cam) for cam in config.CAMERAS]
common_tokens = sorted(set.intersection(*token_sets)) if token_sets else []

if not common_tokens:
    raise RuntimeError('No common tokens found across all cameras. Build dataset first.')

rng = random.Random(SEED)
rng.shuffle(common_tokens)

n = len(common_tokens)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_tokens = common_tokens[:n_train]
val_tokens = common_tokens[n_train:n_train + n_val]
test_tokens = common_tokens[n_train + n_val:]

(SPLIT_DIR / 'train.txt').write_text('\n'.join(train_tokens) + '\n')
(SPLIT_DIR / 'val.txt').write_text('\n'.join(val_tokens) + '\n')
(SPLIT_DIR / 'test.txt').write_text('\n'.join(test_tokens) + '\n')

print(f'Total tokens: {n}')
print(f'Train: {len(train_tokens)}')
print(f'Val:   {len(val_tokens)}')
print(f'Test:  {len(test_tokens)}')
print(f'Splits written to: {SPLIT_DIR}')

## Step 4 - Validate Dataset + DataLoader

Expected shapes:
- Input: `[5, H, W]` = RGB(3) + Depth(1) + Height(1)
- Mask: `[H, W]`

In [ ]:
train_split = config.DATASET_DIR / 'splits' / 'train.txt'
assert train_split.exists(), f'Missing split file: {train_split}'

transform = get_train_transforms(config.IMG_SIZE)
dataset = MultiModalDrivableDataset(
    data_root=config.DATASET_DIR,
    split_file=str(train_split),
    cameras=config.CAMERAS,
    transform=transform,
)

print(f'Dataset length: {len(dataset)}')
x, y = dataset[0]
print('Single sample shapes:')
print('Input:', tuple(x.shape), 'dtype=', x.dtype, 'min=', float(x.min()), 'max=', float(x.max()))
print('Mask :', tuple(y.shape), 'dtype=', y.dtype, 'unique=', torch.unique(y).tolist())

loader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR
)

bx, by = next(iter(loader))
print('Batch shapes:')
print('Inputs:', tuple(bx.shape))
print('Masks :', tuple(by.shape))

In [ ]:
# Visual sanity check of channels from one sample
sample_x, sample_y = dataset[0]

rgb = sample_x[:3].permute(1, 2, 0).numpy()
depth = sample_x[3].numpy()
height = sample_x[4].numpy()
mask = sample_y.numpy()

# Undo normalization for display only
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
rgb_vis = np.clip((rgb * std) + mean, 0.0, 1.0)

fig, axs = plt.subplots(1, 4, figsize=(18, 5))
axs[0].imshow(rgb_vis)
axs[0].set_title('RGB')
axs[0].axis('off')

im1 = axs[1].imshow(depth, cmap='viridis')
axs[1].set_title('Depth Channel')
axs[1].axis('off')
plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(height, cmap='coolwarm')
axs[2].set_title('Height Channel')
axs[2].axis('off')
plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

axs[3].imshow(mask, cmap='gray')
axs[3].set_title('Mask')
axs[3].axis('off')

plt.tight_layout()
plt.show()

## Done

If all checks pass, your dataset is ready for training. Next notebook can load this split and launch training.